In [ ]:
# Section 1: Install Required Libraries
# Run this cell first. It may take 1-2 minutes.
!pip install "datasets==2.19.0" -q
print("Installation complete.")

In [ ]:
# Section 2: Imports and Spark Session Initialization
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, when
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd
import time

spark = SparkSession.builder \
    .appName("Amazon_Grocery_Sentiment_Analysis") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Spark session initialized successfully.")

In [ ]:
# Section 3: Data Acquisition
# Streams the Amazon Reviews 2023 - Grocery and Gourmet Food category
# directly from the Hugging Face API. No full file download required.
from datasets import load_dataset

print("Streaming Grocery and Gourmet Food reviews from Hugging Face...")
start_time = time.time()

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Grocery_and_Gourmet_Food",
    split="full",
    streaming=True
)

# -----------------------------------------------------------------------
# SCALABILITY TEST: Change subset_size to run different experiments.
# Recommended values: 50000 | 100000 | 250000 | 500000
# Start with 50000 to confirm the pipeline works, then increase.
# -----------------------------------------------------------------------
subset_size = 50000

data_list = []
for i, row in enumerate(dataset):
    if i >= subset_size:
        break
    if row.get("text") and row.get("rating"):
        data_list.append({"rating": float(row["rating"]), "text": str(row["text"])})

pdf = pd.DataFrame(data_list)
df = spark.createDataFrame(pdf)

end_time = time.time()
print(f"Data loaded in {end_time - start_time:.2f} seconds. Showing sample:")
df.show(5, truncate=True)

In [ ]:
# Section 4: Data Preprocessing
# - Convert star ratings to binary sentiment labels (positive / negative)
# - Clean review text (lowercase, remove punctuation and numbers)
# - Drop rows with missing values

# Baseline: use star rating as a simple sentiment proxy
# Label 1 = Positive (4-5 stars), Label 0 = Negative (1-2 stars)
# 3-star reviews are excluded as they are ambiguous
df_labeled = df.filter(col("rating") != 3.0) \
               .withColumn("label", when(col("rating") >= 4.0, 1.0).otherwise(0.0))

# Clean text: lowercase, remove punctuation and digits
df_clean = df_labeled.withColumn(
    "clean_text",
    lower(regexp_replace(col("text"), "[^a-zA-Z\\s]", ""))
).dropna(subset=["clean_text", "label"])

print(f"Total records after preprocessing: {df_clean.count()}")
print("Label distribution:")
df_clean.groupBy("label").count().show()

In [ ]:
# Section 5: Spark ML Pipeline — TF-IDF + Logistic Regression
# This pipeline demonstrates distributed processing via Apache Spark.
# Processing this volume of reviews exceeds single-machine capacity;
# Spark enables parallel execution across the cluster.

tokenizer   = Tokenizer(inputCol="clean_text", outputCol="words")
remover     = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF   = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)
idf         = IDF(inputCol="rawFeatures", outputCol="features")
lr          = LogisticRegression(maxIter=10, regParam=0.01,
                                  featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

train_data, test_data = df_clean.randomSplit([0.8, 0.2], seed=42)
print("Data split complete. Ready to train.")
print(f"Training set size (approx): {int(subset_size * 0.8)}")
print(f"Test set size (approx):     {int(subset_size * 0.2)}")

In [ ]:
# Section 6: Model Training
# The Spark ML pipeline is fit on the training data.
# Record this time for your scalability analysis table.

print("Training model...")
train_start = time.time()
model = pipeline.fit(train_data)
train_end = time.time()

training_time = train_end - train_start
print(f"Model training completed in {training_time:.2f} seconds.")
print(f">>> Record this time for subset_size = {subset_size} <<<")

In [ ]:
# Section 7: Model Evaluation
# Evaluate the trained model on the held-out test set.
# Metrics: Accuracy and F1-Score (weighted).

predictions = model.transform(test_data)

# Show sample predictions
print("Sample predictions:")
predictions.select("clean_text", "label", "prediction", "probability").show(5, truncate=True)

# Accuracy
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)

# F1-Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
f1_score = f1_evaluator.evaluate(predictions)

print(f"Test Accuracy : {accuracy * 100:.2f}%")
print(f"Test F1-Score : {f1_score * 100:.2f}%")

In [ ]:
# Section 8: Scalability Analysis
# Record your results in this table after each run.
# Change subset_size in Section 3 and re-run the notebook for each row.

scalability_results = {
    "subset_size":    [50000, 100000, 250000, 500000],
    "training_time":  [None,  None,   None,   None  ],   # fill in after each run
    "accuracy":       [None,  None,   None,   None  ],   # fill in after each run
    "f1_score":       [None,  None,   None,   None  ],   # fill in after each run
}

import pandas as pd
results_df = pd.DataFrame(scalability_results)
print("Scalability Results Table (fill in after each run):")
print(results_df.to_string(index=False))

In [ ]:
# Section 9: Real-Time Sentiment Prediction (Interactive Demo)
# Run this cell and type any product review when prompted, then press Enter.
# The model will instantly predict whether the review is Positive or Negative.
# Great for live demonstrations during your presentation!

import re

def predict_sentiment(review_text):
    """Predict sentiment for a single user-provided review string."""
    # Clean the input the same way as training data
    clean = re.sub(r"[^a-zA-Z\s]", "", review_text.lower()).strip()
    if not clean:
        print("Please enter a non-empty review.")
        return

    # Create a one-row Spark DataFrame
    input_df = spark.createDataFrame([(clean,)], ["clean_text"])

    # Run through the trained pipeline
    result = model.transform(input_df)
    row = result.select("prediction", "probability").collect()[0]

    sentiment  = "POSITIVE ✅" if row["prediction"] == 1.0 else "NEGATIVE ❌"
    confidence = max(row["probability"]) * 100

    print("=" * 60)
    print(f"  Review     : {review_text[:80]}")
    print("-" * 60)
    print(f"  Sentiment  : {sentiment}")
    print(f"  Confidence : {confidence:.1f}%")
    print("=" * 60)

# ── Prompts the user to type a review and press Enter ────────────────────────
user_review = input("Enter a product review: ")
predict_sentiment(user_review)

In [ ]:
# Section 9b: Batch Demo — run several example reviews at once
# Useful for showing the model's range during your presentation.

example_reviews = [
    "Absolutely love this product, will buy again!",
    "Terrible quality, fell apart after one use.",
    "Great value for the price, very happy with my purchase.",
    "Do not waste your money, complete garbage.",
    "Arrived on time and exactly as described, very satisfied.",
]

print("Batch Sentiment Predictions")
print("=" * 60)
for review in example_reviews:
    predict_sentiment(review)
    print()